# Arabic Sentiment: train Colab


In [ ]:
!nvidia-smi | head -n 15

Sat Oct  3 06:56:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/mlops'
import os
import shutil

assert os.path.isdir(DRIVE), f"مش لاقي {DRIVE}"
WORK = '/content/project'
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK + '/data')
for name in ['src', 'configs', 'scripts']:
    shutil.copytree(f'{DRIVE}/{name}', f'{WORK}/{name}')
shutil.copy(f'{DRIVE}/requirements.txt', WORK)
shutil.copytree(f'{DRIVE}/processed', f'{WORK}/data/processed')
os.chdir(WORK)
!ls -la data/processed

Mounted at /content/drive
total 94816
drwx------ 2 root root     4096 Oct  3 06:38 .
drwxr-xr-x 3 root root     4096 Oct  3 06:56 ..
-rw------- 1 root root        0 Oct  2 19:26 .gitkeep
-rw------- 1 root root  9719777 Oct  2 19:54 test.csv
-rw------- 1 root root 77681689 Oct  2 19:54 train.csv
-rw------- 1 root root  9677771 Oct  2 19:54 val.csv


In [ ]:
!pip install -q 'transformers>=4.46' accelerate scikit-learn pandas pyyaml

In [ ]:
import torch

print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

GPU: True Tesla T4


## Train


In [ ]:
import time

t = time.time()
!python -m src.train --max-samples 60000 --epochs 2 --output-dir models/v1
print(f"\nالوقت: {(time.time()-t)/60:.1f} دقيقة")

config.json: 100% 384/384 [00:00<00:00, 2.16MB/s]
tokenizer_config.json: 100% 381/381 [00:00<00:00, 2.16MB/s]
vocab.txt: 100% 825k/825k [00:00<00:00, 19.9MB/s]
tokenizer.json: 100% 2.64M/2.64M [00:00<00:00, 108MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 533kB/s]

model.safetensors: downloading bytes:   2% 10.1M/543M [00:00<00:41, 12.8MB/s]
model.safetensors: downloading bytes:  42% 229M/543M [00:01<00:00, 317MB/s, 17.7MB/s  ]
model.safetensors: downloading bytes:  60% 327M/543M [00:01<00:00, 268MB/s, 27.9MB/s  ]
model.safetensors: downloading bytes:  91% 493M/543M [00:02<00:00, 342MB/s, 40.4MB/s  ]
model.safetensors: reconstructing file:  74% 402M/543M [00:02<00:00, 209MB/s, 25.2MB/s  ]
model.safetensors: downloading bytes: 100% 512M/512M [00:02<00:00, 179MB/s, 45.2MB/s  ]
model.safetensors: reconstructing file: 100% 543M/543M [00:02<00:00, 190MB/s, 48.5MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 6278.05it/s]
[transformers] BertForSequenceClassification LOAD REPOR

## Save in Drive

In [ ]:
import shutil

dst = f'{DRIVE}/models/v1'
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree('models/v1', dst, ignore=shutil.ignore_patterns('checkpoints'))
with open(f'{dst}/metrics.json', 'r', encoding='utf-8') as f:
    print(f.read())